# 🚀 Lab 27: Web Dashboards with Dash

### 📘 Lab Overview
In this lab, you will create an interactive web dashboard using **Dash**, a Python framework for building analytical web applications. You will work with sample sales data to build a dashboard that includes dropdown filters, KPI cards, interactive charts, CSV export, and simulated real-time updates.

### 🎯 Objectives
* Understand the fundamentals of Dash for creating web dashboards.
* Create dashboards with multiple interactive components (dropdowns, graphs).
* Implement callback functions for dynamic interactions.
* Build a complete dashboard with real-time-style updates.
* Export filtered data and apply responsive design best practices.

## 🧰 Prerequisites
* Basic knowledge of Python programming.
* Understanding of pandas for data manipulation.
* Basic understanding of HTML/CSS concepts is helpful but not required.

## ⚙️ Environment Setup

**ELI10:** Before we build our dashboard, we need to bring in special tools (libraries) that help Python talk to web browsers and make pretty charts. We'll use `dash` for the website part and `plotly` for the charts.

In [1]:
# Install necessary libraries for Dash and Plotly in Colab
%pip install dash plotly pandas requests

import dash
from dash import html, dcc, Input, Output, State
import plotly
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import random
import os
from datetime import datetime, timedelta

# Print versions to verify successful installation
print("Dash version:", dash.__version__)
print("Plotly version:", plotly.__version__)
print("Pandas version:", pd.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 46.2 MB/s eta 0:00:00
Dash version: 4.1.0
Plotly version: 5.24.1
Pandas version: 2.2.2


## 📦 Creating Sample Data

**ELI10:** Imagine we have a big store. We need some fake 'sales' info (like what we sold, where, and when) so we have something interesting to show on our dashboard charts.

In [2]:
def generate_sample_data():
    """Generate 1000 rows of random sales data for our dashboard."""
    products = ['Laptops', 'Smartphones', 'Tablets', 'Headphones', 'Cameras']
    regions = ['North America', 'Europe', 'Asia', 'South America', 'Africa']

    data = []
    start_date = datetime(2023, 1, 1)
    random.seed(42) # Ensures the 'random' data is the same every time we run this

    for _ in range(1000):
        date = start_date + timedelta(days=random.randint(0, 365))
        product = random.choice(products)
        region = random.choice(regions)
        sales = random.randint(1000, 50000)
        quantity = random.randint(1, 100)

        data.append({
            'Date': date, 'Product': product, 'Region': region,
            'Sales': sales, 'Quantity': quantity
        })

    df = pd.DataFrame(data)
    df['Date'] = pd.to_datetime(df['Date'])
    return df

# Generate and save to CSV so our app can read it like a real database
df = generate_sample_data()
df.to_csv("sales_data.csv", index=False)

print("Sample data generated and saved to sales_data.csv!")
display(df.head())

Sample data generated and saved to sales_data.csv!


,Date,Product,Region,Sales,Quantity
0,2023-11-24,Laptops,North America,49598,36
1,2023-05-06,Smartphones,Europe,49265,14
2,2023-12-13,Cameras,North America,39698,55
3,2023-01-17,Laptops,North America,15328,30
4,2023-09-16,Cameras,North America,37781,26


## 🧱 Building the Dashboard Layout

**ELI10:** Now we define the 'skeleton' of our website. We tell Dash where to put the title, the dropdown menus, and the boxes where the charts will go.

In [3]:
# Re-load data and prepare a 'Month' column for time-series charts
df = pd.read_csv("sales_data.csv")
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.to_period("M").astype(str)

# Initialize the Dash app
app = dash.Dash(__name__)

# Define the visual structure (Layout)
app.layout = html.Div([
    # Title
    html.H1("Interactive Sales Dashboard", style={"textAlign": "center", "color": "#2c3e50"}),

    # Filter Panel
    html.Div([
        html.Div([
            html.Label("Select Product:"),
            dcc.Dropdown(id="product-dropdown",
                         options=[{"label": "All Products", "value": "all"}] +
                                 [{"label": p, "value": p} for p in sorted(df["Product"].unique())],
                         value="all")
        ], style={"width": "30%", "display": "inline-block", "padding": "10px"}),

        html.Div([
            html.Label("Select Region:"),
            dcc.Dropdown(id="region-dropdown",
                         options=[{"label": "All Regions", "value": "all"}] +
                                 [{"label": r, "value": r} for r in sorted(df["Region"].unique())],
                         value="all")
        ], style={"width": "30%", "display": "inline-block", "padding": "10px"}),

        html.Div([
            html.Label("Select Metric:"),
            dcc.Dropdown(id="metric-dropdown",
                         options=[{"label": "Sales Amount", "value": "Sales"},
                                  {"label": "Quantity Sold", "value": "Quantity"}],
                         value="Sales")
        ], style={"width": "30%", "display": "inline-block", "padding": "10px"})
    ], style={"backgroundColor": "#f8f9fa", "borderRadius": "5px", "margin": "10px"}),

    # KPI Cards Container
    html.Div(id="kpi-cards", style={"display": "flex", "justifyContent": "space-around"}),

    # Charts Grid
    html.Div([
        html.Div([dcc.Graph(id="time-series-chart")], style={"width": "50%", "display": "inline-block"}),
        html.Div([dcc.Graph(id="bar-chart")], style={"width": "50%", "display": "inline-block"})
    ]),

    html.Div([
        html.Div([dcc.Graph(id="pie-chart")], style={"width": "50%", "display": "inline-block"}),
        html.Div([dcc.Graph(id="scatter-plot")], style={"width": "50%", "display": "inline-block"})
    ]),

    # Export and Interval Components (Advanced Features)
    html.Div([
        html.Button("Export Filtered Data", id="export-button", style={"marginTop": "20px"}),
        dcc.Download(id="download-dataframe-csv"),
        html.Div(id="export-status")
    ], style={"textAlign": "center"}),

    dcc.Interval(id="interval-component", interval=10*1000, n_intervals=0, disabled=True),
    html.Button("Toggle Real-time Updates", id="realtime-button", style={"margin": "10px"}),
    html.Div(id="realtime-status")
])

## 🔁 Dash Callbacks

**ELI10:** Callbacks are like the 'brain' of the dashboard. When you click a button or pick something from a dropdown, the callback notices the change and tells the charts to redraw themselves with the new data.

In [4]:
# Helper function to filter data based on user selection
def filter_data(product, region):
    filtered_df = df.copy()
    if product != "all":
        filtered_df = filtered_df[filtered_df["Product"] == product]
    if region != "all":
        filtered_df = filtered_df[filtered_df["Region"] == region]
    return filtered_df

# Helper function to create a KPI card HTML component
def create_kpi_card(title, value, color="#3498db", prefix=""):
    return html.Div([
        html.H4(title, style={"margin": "0"}),
        html.H2(f"{prefix}{value:,.0f}" if isinstance(value, (int, float)) else str(value),
                style={"color": color})
    ], style={"padding": "20px", "border": "1px solid #ddd", "borderRadius": "5px", "width": "20%"})

# Callback 1: Update KPI Cards
@app.callback(
    Output("kpi-cards", "children"),
    [Input("product-dropdown", "value"), Input("region-dropdown", "value")]
)
def update_kpis(prod, reg):
    f_df = filter_data(prod, reg)
    return [
        create_kpi_card("Total Sales", f_df["Sales"].sum(), "#e74c3c", "$"),
        create_kpi_card("Quantity", f_df["Quantity"].sum(), "#2ecc71"),
        create_kpi_card("Avg Order", f_df["Sales"].mean(), "#f39c12", "$")
    ]

# Callback 2: Update Time Series Chart
@app.callback(
    Output("time-series-chart", "figure"),
    [Input("product-dropdown", "value"), Input("region-dropdown", "value"), Input("metric-dropdown", "value")]
)
def update_time_chart(prod, reg, metric):
    f_df = filter_data(prod, reg)
    monthly = f_df.groupby("Month")[metric].sum().reset_index()
    fig = px.line(monthly, x="Month", y=metric, title=f"{metric} Trend")
    return fig.update_layout(title_x=0.5)

# Callback 3: Update Bar Chart
@app.callback(
    Output("bar-chart", "figure"),
    [Input("product-dropdown", "value"), Input("region-dropdown", "value"), Input("metric-dropdown", "value")]
)
def update_bar(prod, reg, metric):
    f_df = filter_data(prod, reg)
    group_col = "Product" if prod == "all" else "Region"
    data = f_df.groupby(group_col)[metric].sum().reset_index()
    return px.bar(data, x=group_col, y=metric, title=f"{metric} by {group_col}", color=metric)

# Callback 4: Update Pie Chart
@app.callback(
    Output("pie-chart", "figure"),
    [Input("product-dropdown", "value"), Input("region-dropdown", "value"), Input("metric-dropdown", "value")]
)
def update_pie(prod, reg, metric):
    f_df = filter_data(prod, reg)
    group_col = "Region" if reg == "all" else "Product"
    data = f_df.groupby(group_col)[metric].sum().reset_index()
    return px.pie(data, values=metric, names=group_col, title=f"{metric} Distribution")

# Callback 5: Update Scatter Plot
@app.callback(
    Output("scatter-plot", "figure"),
    [Input("product-dropdown", "value"), Input("region-dropdown", "value")]
)
def update_scatter(prod, reg):
    f_df = filter_data(prod, reg)
    return px.scatter(f_df, x="Sales", y="Quantity", color="Product", size="Sales", title="Sales vs Quantity")

## ⬇️ Export and Live Updates

**ELI10:** We can also add buttons to download our data as a file (CSV) or toggle a timer that refreshes our charts every few seconds, just like a real stock market dashboard!

In [5]:
# Callback 6: Export Data to CSV
@app.callback(
    Output("download-dataframe-csv", "data"),
    Output("export-status", "children"),
    Input("export-button", "n_clicks"),
    [State("product-dropdown", "value"), State("region-dropdown", "value")],
    prevent_initial_call=True
)
def export_csv(n, prod, reg):
    f_df = filter_data(prod, reg)
    return dcc.send_data_frame(f_df.to_csv, f"sales_{prod}_{reg}.csv", index=False), "Export Successful!"

# Callback 7: Real-time Update Toggle
@app.callback(
    Output("interval-component", "disabled"),
    Output("realtime-status", "children"),
    Input("realtime-button", "n_clicks")
)
def toggle_updates(n):
    if n and n % 2 == 1:
        return False, "Real-time updates: ON"
    return True, "Real-time updates: OFF"

## ⚡ Running the Dashboard

**Note:** In Colab, we use `jupyter_mode="external"` to provide a link to the dashboard. If you are running this locally, you can use `inline`.

In [6]:
# Run the app. In Colab, this generates a link to view your dashboard.
# We set debug=False for more stable execution in a notebook environment.
if __name__ == '__main__':
    app.run(jupyter_mode="external", debug=False, port=8050)

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit


## ✅ Verification Checklist
- [ ] Data generated and `sales_data.csv` exists.
- [ ] Dashboard title and dropdowns are visible.
- [ ] KPI cards update when changing Product or Region.
- [ ] Charts redraw based on filter selection.
- [ ] CSV Export button triggers a file download.

## 🛠 Troubleshooting
* **Port Busy:** If port 8050 is taken, change the `port` parameter in `app.run` to `8051`.
* **No Chart:** Check if your dropdown values match the values in your CSV (case-sensitive).
* **Browser Block:** Ensure pop-ups/external links are allowed by your browser in Colab.

## 📚 Key Takeaways
* **Dash Layout:** The visual structure built with HTML and Core components.
* **Callbacks:** The logic linking user input to app outputs.
* **Interactivity:** Using Plotly Express to create dynamic, hoverable visualizations.

### 🎓 Conclusion
You've built a professional business dashboard! This tool allows stakeholders to filter massive datasets instantly and visualize trends, which is a core skill for any Data Scientist or Business Analyst.